In [1]:
import os
import pandas as pd

import matplotlib.pyplot as plt
from numba.np.math.numbers import NAN

from data.visualize import plot_stations_on_dem, markers_from_stations_table



import peakweather

print("PeakWeather version ",peakweather.__version__)

from peakweather import PeakWeatherDataset

#Load Local Dataset
# Exclude: rain gauge stations (142) & sunshine variable
ds = PeakWeatherDataset(root="/Users/aureliedejong/Documents/ETH/_DAS Project/PeakWeatherDataset",
                        parameters = ["temperature",
                                      "pressure",
                                      "humidity",
                                      "wind_speed",
                                      "wind_gust",
                                      "wind_direction",
                                      "precipitation",
                                      ],
                        extended_topo_vars="DEM",
                        imputation_method=None,
                        compute_uv = True,
                        station_type="meteo_station")

PeakWeather version  0.2.1


### Spatial Embedding

In [2]:
print("Check number of stations:", len(ds.stations_table))

Check number of stations: 156


In [3]:
ds.stations_table.columns

Index(['station_name', 'latitude', 'longitude', 'station_height',
       'swiss_easting', 'swiss_northing', 'ASPECT_2000M_SIGRATIO1',
       'WE_DERIVATIVE_2000M_SIGRATIO1', 'TPI_2000M',
       'SN_DERIVATIVE_10000M_SIGRATIO1', 'dem',
       'SN_DERIVATIVE_2000M_SIGRATIO1', 'SLOPE_10000M_SIGRATIO1',
       'ASPECT_10000M_SIGRATIO1', 'SLOPE_2000M_SIGRATIO1', 'STD_2000M',
       'STD_10000M', 'TPI_10000M', 'WE_DERIVATIVE_10000M_SIGRATIO1',
       'station_type'],
      dtype='str')

In [4]:
coordinates = ds.stations_table[['swiss_easting', 'swiss_northing']] # spatial embedding (lat, lon) → [sin(Bx), cos(Bx)]

#Call one station⁄
ds.stations_table.drop(columns=["latitude", "longitude", "station_name", "station_type"]).loc[station]

NameError: name 'station' is not defined

### Temporal Embedding

In [ ]:
station = 'ABO'
ds.get_observations(station).head()

In [ ]:
print("Start recording", ds.get_observations(station).index[0].date())
print("End recording", ds.get_observations(station).index[-1].date())

In [ ]:
ds.get_observations(station).index[-1] - ds.get_observations(station).index[0]

In [ ]:
3207/365

In [ ]:
ds.get_observations(station).index

### Variables

In [ ]:
ds.get_observations(station).iloc[0]

In [ ]:
ds.available_parameters

#### Explore Variables Data

In [ ]:
ds_day = PeakWeatherDataset(root="/Users/aureliedejong/Documents/ETH/_DAS Project/PeakWeatherDataset",
                        parameters = ["temperature",
                                      "pressure",
                                      "humidity",
                                      "wind_speed",
                                      "wind_gust",
                                      "wind_direction",
                                      "precipitation",
                                      ],
                        freq = "d",
                        station_type="meteo_station")

In [ ]:
df, mask = ds_day.get_observations(
    #first_date="2024-08-02 16:32",
    #last_date="2024-08-06 23:26",
    return_mask=True,
)

long_df = mask.stack(0).rename_axis(["datetime","nat_abbr"]).reset_index()
df_time = long_df.groupby(["datetime"]).sum().drop(columns=["nat_abbr"]).reset_index()

variables = list(ds_day.parameters)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for v in variables:
    ax.plot(
        df_time["datetime"],
        df_time[v].rolling(30).mean(),
        label=v
    )

ax.set_title("Station availability (Monthly average)")
ax.set_ylabel("Number of stations")
ax.set_xlabel("Datetime")

# legend on the right side
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

ax.grid(True)

plt.tight_layout()
plt.show()